In [0]:
from pyspark.sql.functions import col, sum, avg, count, when, round, upper

In [0]:
from pyspark.sql.functions import coalesce, lit

print("\n" + "="*60)
print("CREATING DATA CUBE")
print("="*60)
data_cube = (
    spark.table("fact.fact_invoice_line_items")
    .alias("f")
    .join(
        spark.table("dim.dim_customers").alias("c"),
        col("f.customer_key") == col("c.customer_key"),
        "left"
    )
    .join(
        spark.table("dim.dim_products").alias("p"),
        col("f.product_key") == col("p.product_key"),
        "left"
    )
    .join(
        spark.table("dim.dim_dates").alias("d"),
        col("f.date_key") == col("d.date_key"),
        "left"
    )

    .join(
        spark.table("fact.fact_invoices").alias("fi"),
        col("f.invoice_id") == col("fi.invoice_id"),
        "left"
    )
    .join(
        spark.table("dim.dim_regions").alias("r"),
        col("fi.region_key") == col("r.region_key"),
        "left"
    )
  
    .join(
        spark.table("dim.dim_exchange_rates").alias("ex"),
        (col("fi.currency") == col("ex.currency")) & 
        (col("f.date_key") == col("ex.date")),
        "left"
    )
    .groupBy(
       
        coalesce(col("c.customer_id"), lit("none")).alias("customer_id"),
        coalesce(col("c.customer_name"), lit("none")).alias("customer_name"),
        coalesce(col("c.customer_type"), lit("none")).alias("customer_type"),
        upper(coalesce(col("c.country"), lit("none"))).alias("customer_country"),
     
        coalesce(col("p.product_id"), lit("none")).alias("product_id"),
        coalesce(col("p.product_name"), lit("none")).alias("product_name"),
        coalesce(col("p.category"), lit("none")).alias("category"),
       
        coalesce(col("r.region_code"), lit("none")).alias("region_code"),
        coalesce(col("r.region_name"), lit("none")).alias("region_name"),
        upper(coalesce(col("r.country"), lit("none"))).alias("region_country"),
      
        coalesce(col("d.year"), lit(0)).alias("year"),
        coalesce(col("d.quarter"), lit(0)).alias("quarter"),
        coalesce(col("d.month"), lit(0)).alias("month"),
        coalesce(col("d.month_name"), lit("none")).alias("month_name"),
       
        coalesce(col("fi.currency"), lit("none")).alias("currency")
    )
    .agg(
       
        coalesce(sum("f.line_total"), lit(0)).alias("total_revenue"),
        coalesce(sum("f.quantity"), lit(0)).alias("total_quantity"),
        coalesce(sum("f.profit"), lit(0)).alias("total_profit"),
        count("f.invoice_line_id").alias("transaction_count"),
        coalesce(avg("f.unit_price"), lit(0)).alias("avg_unit_price"),
        coalesce(avg("f.discount"), lit(0)).alias("avg_discount"),
      
        coalesce(sum(col("f.line_total") * col("ex.rate_to_usd")), lit(0)).alias("total_revenue_usd"),
        coalesce(sum(col("f.profit") * col("ex.rate_to_usd")), lit(0)).alias("total_profit_usd")
    )
    .withColumn(
        "profit_margin_pct",
        when(col("total_revenue") > 0, 
             round((col("total_profit") / col("total_revenue")) * 100, 2)
        ).otherwise(0)
    )
)

data_cube.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact.sales_cube")
display(spark.table("fact.sales_cube").limit(10))